# 03 Lasso 回归

依赖安装说明：`pip install numpy matplotlib scikit-learn`

Lasso 是带 L1 正则化的线性回归。它最重要的特点是：可以把一部分系数压到精确的 0，因此常被用来做特征选择。


## 1. 数学逻辑

Lasso 的目标函数是：

$$L(w)=\frac{1}{2n}\sum_{i=1}^{n}(y_i-X_iw)^2 + \alpha\sum_{j=1}^{d}|w_j|$$

L1 正则项使用绝对值：

$$||w||_1 = \sum_j |w_j|$$

和 Ridge 的 L2 不同，L1 的几何形状更容易让最优解落在坐标轴上，所以一些系数会变成 0。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

n, d = 150, 12
X = np.random.normal(size=(n, d))
true_w = np.array([4.0, -3.0, 2.0] + [0.0] * (d - 3))
y = X @ true_w + np.random.normal(scale=1.0, size=n)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


In [ ]:
# 从零实现：一维 soft-thresholding 是 Lasso 的核心直觉
# 如果普通梯度想把系数推得不够大，L1 会直接把它压成 0。

def soft_threshold(z, gamma):
    if z > gamma:
        return z - gamma
    if z < -gamma:
        return z + gamma
    return 0.0

for z in [-2, -0.5, 0.2, 1.5, 3.0]:
    print(f'z={z: .1f} -> soft_threshold(z, 1.0)={soft_threshold(z, 1.0): .1f}')

# 一个简化版坐标下降：逐个更新每个系数
alpha = 0.08
w = np.zeros(d)
b = y_train.mean()
Xc = X_train_s
yc = y_train - b

for epoch in range(80):
    for j in range(d):
        residual = yc - Xc @ w + Xc[:, j] * w[j]
        rho = np.mean(Xc[:, j] * residual)
        w[j] = soft_threshold(rho, alpha) / (np.mean(Xc[:, j] ** 2) + 1e-12)

print('真实系数:', true_w)
print('从零 Lasso 系数:', np.round(w, 3))


In [ ]:
# sklearn 实战
ols = LinearRegression().fit(X_train_s, y_train)
lasso = Lasso(alpha=0.08, max_iter=10000).fit(X_train_s, y_train)

for name, model in [('LinearRegression', ols), ('Lasso', lasso)]:
    pred = model.predict(X_test_s)
    print(name)
    print('  weights:', np.round(model.coef_, 3))
    print('  非零系数数量:', np.sum(np.abs(model.coef_) > 1e-8))
    print('  MSE:', round(mean_squared_error(y_test, pred), 3))

alphas = np.logspace(-3, 0, 50)
coefs = np.array([Lasso(alpha=a, max_iter=10000).fit(X_train_s, y_train).coef_ for a in alphas])
plt.plot(alphas, coefs)
plt.xscale('log')
plt.title('alpha 越大，Lasso 会把更多系数压到 0')
plt.xlabel('alpha')
plt.ylabel('coefficient')
plt.show()


## 2. 常见误区

- Lasso 对特征尺度非常敏感，通常必须标准化。
- 当多个特征高度相关时，Lasso 可能随机留下其中一个，解释时要小心。
- Lasso 做的是线性特征选择，不代表被压成 0 的变量在真实世界中一定无意义。

## 3. 小实验

- 改 `alpha`，观察非零系数数量。
- 增加相关特征，看 Lasso 选择哪个。
- 对比 Ridge：Ridge 更平滑，Lasso 更稀疏。
